# Validación: Marino et al. (2024), turbina NTNU con disco actuador

Réplica del caso de validación de *Modelling Wind Turbines via Actuator Lines in High-Order h/p Solvers* (§3, sin torre ni góndola) con el disco actuador de la PR #121. Hay dos campañas con el mismo `.control` y la misma malla; solo cambia el modelo:

- **A (`pressure`):** Δp uniforme, fijado al C_T experimental de 0.9.
- **B (`bladeElement`):** la pala del tutorial (30 secciones y polares S826), que predice C_T y C_P.

Cada campaña tiene su raíz (`model_A`, `model_B`), porque GANDALF guarda una campaña por raíz. Las decisiones están en `wind_turbine_marino2024/<model>/metadata/LEEME.md`.

In [ ]:
import os
import pandas as pd
from FotR import GANDALF, FRODO

base = '/home/m.jaraiz/Documentos/COMAC/wind_turbine_marino2024'
sources = os.path.join(base, 'sources')
mesh = '/home/m.jaraiz/Documentos/COMAC/wind_turbine_ansa/sources/Wind_turbine_disk_fluid_cgns_g2_mesh.h5'
utils = '/home/m.jaraiz/Documentos/utils_H3d'
repo = '~/repos/horses3d-gpu-ad'     # rama arr/actuator_disk (PR #121), compilada para GPU

models = {'A': 'pressure', 'B': 'bladeElement'}

## Carpetas

- **Campaña A:** p2 empieza desde el flujo uniforme y va de 0 a 30, con estadísticas desde 20. Cada orden siguiente reinicia y avanza 10 unidades: 5 de transitorio y 5 de estadísticas.
- **Campaña B (23/11/2026):** corre en Flexo (CPU). Su p2 reinicia desde el autoguardado de A en t₀ = 15 (`adA_g2p2_0000215483.hsol`). A ya es estacionario desde t ≈ 12, así que B se ahorra la formación de la estela.
  - p2: de 15 a 27, con estadísticas desde 22.
  - p3, p4 y p5: +10 cada uno (37, 47 y 57), con estadísticas desde 32, 42 y 52.
  - El fichero de A se copia (no se mueve) a `RESTART/` del caso B. HORSES3D toma el tiempo y la iteración del fichero.
- **Estadísticas:** `starting time` está dentro de `#define statistics`, y el ring lo cambia por orden con la clave `'#define statistics: starting time'`.
- **`overwrite`:** se generó con `False`. B se regeneró con `True` el 23/11/2026, antes de lanzarlo; `overwrite` nunca toca `RESULTS/` ni `LOG/`.

In [ ]:
import shutil

stats = '#define statistics: starting time'
restart_B = 'RESTART/adA_g2p2_0000215483.hsol'          # A en t = 15
p_values = {
    'A': {3: {'final time': 40.0, stats: 35.0},
          4: {'final time': 50.0, stats: 45.0},
          5: {'final time': 60.0, stats: 55.0}},
    'B': {2: {'restart': True, 'restart file name': f'"{restart_B}"', 'final time': 27.0, stats: 22.0},
          3: {'final time': 37.0, stats: 32.0},
          4: {'final time': 47.0, stats: 42.0},
          5: {'final time': 57.0, stats: 52.0}},
}

gdfs = {}
for name, model in models.items():
    gdf = GANDALF(root_dir=os.path.join(base, f'model_{name}'), ring='horses3d')
    gdf.define_cases(method='external', external_dataframe=pd.DataFrame({'tsr': [6.0]}))
    print(gdf.generate_folders(
        control_template=os.path.join(sources, 'wind_turbine_marino.control'),
        mesh_path=mesh,
        tag=f'ad{name}',
        folder_fmt=f'marino_{name}_tsr_{{tsr:.1f}}',
        p_levels=[2, 3, 4, 5],
        control_values={'actuator disk model': f'"{model}"'},
        p_values=p_values[name],
        extra_dirs=[os.path.join(sources, name, 'ActuatorDef')],
        run_script=os.path.join(utils, 'run.sh'),
        fsp=False,
        overwrite=False,
    ))
    gdfs[name] = gdf

case_A = os.path.join(base, 'model_A', 'outputs', 'marino_A_tsr_6.0')
case_B = os.path.join(base, 'model_B', 'outputs', 'marino_B_tsr_6.0')
target = os.path.join(case_B, restart_B)
if not os.path.exists(target):
    os.makedirs(os.path.dirname(target), exist_ok=True)
    shutil.copy2(os.path.join(case_A, 'RESULTS', os.path.basename(restart_B)), target)
print(target, os.path.getsize(target))

## Lanzamiento

- **Campaña A, en nibbler:** todos los órdenes con 2 GPU y 4 CPU por GPU. Lanzada el 23/11/2026 (jobs 4373–4376).
- **Campaña B, en Flexo (CPU):** la celda se ejecuta **desde Flexo**, con el binario gfortran de `~/repos/horses3d-gpu-ad-cpu` (`build_solver.sh`). Los nodos y núcleos se deciden allí con `sinfo`.
  - p5 puede rozar las 48 h de `script_flexo_cpu.slurm`. `SBATCH_TIMELIMIT` tiene prioridad sobre `#SBATCH --time`, y `submit_cases` hereda el entorno; antes hay que mirar el MaxTime de la partición (`sinfo -o "%P %l"`).
- **Encadenado:** cada orden espera a que el anterior termine bien (`afterok`).

In [ ]:
launch_A = dict(arch='GPU', node='n001', ngpu=2, cpus_per_gpu=4)
print(gdfs['A'].submit_cases([2, 3, 4, 5], repo, **launch_A)[['p', 'status', 'depends_on', 'command']].to_string())

In [ ]:
done = gdfs['A'].submit_cases([2, 3, 4, 5], repo, **launch_A, dry_run=False)
print(done[['p', 'status', 'job_id', 'depends_on', 'message']].to_string())

In [ ]:
# Campaña B: SOLO desde Flexo
repo_cpu = '~/repos/horses3d-gpu-ad-cpu'
launch_B = dict(arch='CPU', node='n[001-007]', ncpus=320)    # ajustar con sinfo en Flexo
# os.environ['SBATCH_TIMELIMIT'] = '96:00:00'                # si la partición lo permite
print(gdfs['B'].submit_cases([2, 3, 4, 5], repo_cpu, **launch_B)[['p', 'status', 'depends_on', 'command']].to_string())
# gdfs['B'].submit_cases([2, 3, 4, 5], repo_cpu, **launch_B, dry_run=False)

## Coste con p6 (Fig. 3)

El coste de p2 a p5 sale del `.residuals` de cada orden, que guarda el tiempo del solver en cada iteración. Para p6, `benchmarks_p6/bench_p6.slurm` ejecuta 200 iteraciones con 1 GPU para A, B y sin disco.

In [ ]:
import subprocess
print(subprocess.run(['sbatch', 'bench_p6.slurm'], cwd=os.path.join(base, 'benchmarks_p6'),
                     capture_output=True, text=True).stdout)

## Seguimiento

In [ ]:
for name in models:
    root = os.path.join(base, f'model_{name}')
    db = FRODO(root_dir=root, format='HORSES3D')
    print(name, db.df_state[['case', 'p_available', 'p_finished', 'state']].to_string(index=False))
    ct = os.path.join(root, 'outputs', f'marino_{name}_tsr_6.0', 'RESULTS', f'ad{name}_g2p2_Actuator_Disk_CP_CT_disk_001.dat')
    if os.path.exists(ct):
        print(pd.read_csv(ct, sep=r'\s+').tail(3))